# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Load environment variables from a .env file into the notebook environment to keep sensitive information secure.
%load_ext dotenv
%dotenv 

In [2]:
# Working with Dask DataFrames for scalable data processing.
import dask.dataframe as dd


/opt/anaconda3/envs/dsi_participant/lib/python3.12/site-packages/dask/dataframe/_pyarrow_compat.py:17: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 13.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import os
import sys
from glob import glob

# Ensure the SRC_DIR environment variable is set and add it to sys.path for module imports.
sys.path.append(os.getenv('SRC_DIR'))

# Importing a custom logger from the utils module to log important information.
from utils.logger import get_logger
_logs = get_logger(__name__)


# Load the environment variable PRICE_DATA for the data path to the parquet files.
PRICE_DATA = os.getenv("PRICE_DATA")      

# Using the glob module to find all parquet files in the PRICE_DATA directory and its subdirectories recursively.
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)

# Log the loading of environment variables and the number of parquet files found.
_logs.info(f"Found {len(parquet_files)} parquet files in PRICE_DATA directory.")

2025-09-26 22:30:12,370, 1997363655.py, 18, INFO, Environment variables loaded and import path updated.
2025-09-26 22:30:12,371, 1997363655.py, 19, INFO, Found 2010 parquet files in PRICE_DATA directory.


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
# Read all parquet files into a Dask DataFrame and set the index to 'ticker'.
dd_px = dd.read_parquet(parquet_files).set_index("ticker")

# Create lagged features and calculate returns and high-low range for each ticker.
dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1     = x['Close'].shift(1),                      # Lagged Close price
        Adj_Close_lag_1 = x['Adj Close'].shift(1),                  # Lagged Adjusted Close price
        returns         = x['Close'] / x['Close'].shift(1) - 1,     # Daily returns
        hi_lo_range     = x['High'] - x['Low']                      # High-Low range
    )
)

# Log the completion of data loading and feature creation.
_logs.info("Loaded %d parquet files into Dask DataFrame.", len(parquet_files))
_logs.info("Created lags for variables Close and Adj_Close, returns, and hi_lo_range features for each ticker.")


/var/folders/5h/h1f0cx75553g3c45m_p5b0k80000gn/T/ipykernel_9557/2051851663.py:5: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
2025-09-26 22:33:13,996, 2051851663.py, 15, INFO, Loaded 2010 parquet files into Dask DataFrame.
2025-09-26 22:33:13,997, 2051851663.py, 16, INFO, Created lags for variables Close and Adj_Close, returns, and hi_lo_range features for each ticker.


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [7]:
# Converting to pandas to calculate the rolling mean.
dd_rets = dd_feat.compute()

# Calculating the 10-day moving average for returns using .rolling(10).mean()
dd_rets['returns_moving_average_10'] = dd_rets['returns'].rolling(10).mean()

# Log that the computation is done
_logs.info(f"10-day moving average computed. Final DataFrame shape: {dd_rets.shape}")

2025-09-26 22:36:25,524, 1187659604.py, 8, INFO, 10-day moving average computed. Final DataFrame shape: (230273, 14)


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

In [ ]:
# It was not to convert to pandas to calulate the rolling mean.
# We could have used Dask's rolling method directly on the Dask DataFrame.

# It would be more efficient to use Dask's rolling method directly on the Dask DataFrame.
# Using Dask partitions and parallelism would help handle larger datasets without running into memory issues.
# The current approach requires loading the entire dataset into memory which may not be feasible for very large datasets.


## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [X] Created a branch with the correct naming convention.
- [X] Ensured that the repository is public.
- [X] Reviewed the PR description guidelines and adhered to them.
- [X] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.